<a href="https://colab.research.google.com/github/mariaaapetrovskaya/compling_nlp_course/blob/main/module1_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [ ]:
!mkdir -p data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
!unzip -o data/lenta.txt.zip -d data/

--2026-10-29 15:33:19--  https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip [following]
--2026-10-29 15:33:20--  https://raw.githubusercontent.com/mannefedov/compling_nlp_hse_course/master/data/lenta.txt.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5723675 (5.5M) [application/zip]
Saving to: ‘data/lenta.txt.zip’

lenta.txt.zip       100%[===================>]   5.46M  --.-KB/s    in 0.02s   

2026-10-29 15:33:20 (345 MB/s) - ‘data/lenta.txt.zip’

In [ ]:
import re
from collections import Counter

text = open('data/lenta.txt', encoding='utf-8').read()

pattern = r'(?<!\w)(?=\w*[А-Яа-яЁё])(?=\w*[A-Za-z])\w+(?!\w)'

words = re.findall(pattern, text)
counter = Counter(words)

counter.most_common(10)

[('Яndex', 38),
 ('cообщает', 7),
 ('Yтро', 6),
 ('СNN', 6),
 ('Cотрудники', 6),
 ('Аssociated', 6),
 ('Cтатье', 6),
 ('cовета', 5),
 ('Сentral', 4),
 ('Рress', 4)]

Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) примеры слов, в которых совмещается кириллица и латиница. Найдите самое частотное такое слово и вставьте его в форму как ответ.

### Задание 2 (1 балл)

Дана последовательность слов:

`об этом сообщает агентство интерфакс со ссылкой на пресс-службу`

Приведите весь текст корпуса `data/lenta.txt` к нижнему регистру и сделайте простую токенизацию: разбейте все по пробелам, удалите пунктуацию по краям токенов (`strip`) и отфильтруйте пустые строки, оставшиеся после этих шагов.

По частотам из корпуса рассчитайте для каждого триграмма в последовательности условную вероятность P(w3 | w1 w2) = count(w1 w2 w3) / count(w1 w2). Вставьте в форму триграмм с самой низкой вероятностью (три слова через пробел).

*если где-то получается деление но ноль, то пропустите это сочетание

In [ ]:
import string
from collections import Counter

text = open('data/lenta.txt', encoding='utf-8').read().lower()

punctuation = string.punctuation + '«»„“”‘’…—–−'

tokens = [
    token.strip(punctuation)
    for token in text.split()
]
tokens = [token for token in tokens if token]


bigrams = Counter(
    (tokens[i], tokens[i + 1])
    for i in range(len(tokens) - 1)
)

trigrams = Counter(
    (tokens[i], tokens[i + 1], tokens[i + 2])
    for i in range(len(tokens) - 2)
)

sequence = 'об этом сообщает агентство интерфакс со ссылкой на пресс-службу'.split()

results = []

for i in range(len(sequence) - 2):
    w1, w2, w3 = sequence[i:i + 3]
    denominator = bigrams[(w1, w2)]
    numerator = trigrams[(w1, w2, w3)]

    if denominator == 0:
        continue

    probability = numerator / denominator
    results.append((probability, w1, w2, w3, numerator, denominator))

for result in results:
    print(result)

if results:
    print('\n', ' '.join(min(results, key=lambda x: x[0])[1:4]))

(0.15990990990990991, 'об', 'этом', 'сообщает', 284, 1776)
(0.0763888888888889, 'этом', 'сообщает', 'агентство', 22, 288)
(0.020348837209302327, 'сообщает', 'агентство', 'интерфакс', 7, 344)
(0.07407407407407407, 'агентство', 'интерфакс', 'со', 2, 27)
(0.967741935483871, 'интерфакс', 'со', 'ссылкой', 180, 186)
(0.9920063948840927, 'со', 'ссылкой', 'на', 1241, 1251)
(0.11146752205292702, 'ссылкой', 'на', 'пресс-службу', 139, 1247)

 сообщает агентство интерфакс


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=8)`.

Посчитайте эмбеддинги этих текстов моделью `cointegrated/rubert-tiny2` (без префиксов). Рассчитайте косинусную близость всех со всеми (для каждой пары текстов) и найдите самую близкую пару текстов.

Вставьте в форму первые слова обоих текстов через пробел (например, `Холдинг Американская`, порядок не важен).

!NB: Тексты должны быть разными

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

url = (
    "https://github.com/mannefedov/compling_nlp_hse_course/"
    "raw/master/data/lenta_40k.csv.zip"
)

lenta = pd.read_csv(url)
lenta = lenta.sample(500, random_state=8)

texts = lenta['text'].fillna('').astype(str).tolist()

model = SentenceTransformer('cointegrated/rubert-tiny2')

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

similarities = cosine_similarity(embeddings)


np.fill_diagonal(similarities, -np.inf)


i, j = np.unravel_index(
    np.argmax(similarities),
    similarities.shape
)

def first_word(text):
    words = text.split()
    return words[0] if words else ''

print('1 слово первого текста:', first_word(texts[i]))
print('1 слово второго текста:', first_word(texts[j]))
print('Косинусн близость:', similarities[i, j])

print('\nПервый текст:\n', texts[i][:500])
print('\nВторой текст:\n', texts[j][:500])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

...

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

1 слово первого текста: Нападающий
1 слово второго текста: Футболист
Косинусн близость: 0.95381546

Первый текст:
 Нападающий «Бастии» Брандао появился на заседании Федерации футбола Франции вместе со своей матерью. Там обсуждалась его апелляция на решение дисциплинарного комитета французской Профессиональной футбольной лиги о дисквалификации Брандао на шесть месяцев за то, что он ударил игрока ПСЖ Тиаго Мотту. Об этом сообщает Reuters. 16 августа после окончания матча второго тура чемпионата Франции между «Бастией» и ПСЖ, прошедшем в Париже, в подтрибунном помещении Брандао дождался, когда мимо него пройдет

Второй текст:
 Футболист французского клуба «Бастия» Брандао, сломавший нос игроку ПСЖ Тьягу Мотте, решением дисциплинарного комитета французской Профессиональной футбольной лиги дисквалифицирован на полгода. Об этом сообщает французское издание L'Equpe. Срок дисквалификации игрока де-юре начался 22 августа и истечет 22 февраля. Инцидент произошел 16 августа в подтрибунном помещен